### Middleware

Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:
- Tracking agent behavior with logging, analytics, and debugging.
- Transforming prompts, tool selection, and output formatting.
- Adding retries, fallbacks, and early termination logic.
- Applying rate limits, guardrails, and PII detection.

In [3]:
import os
from dotenv import load_dotenv
load_dotenv()


os.environ["GEMINI_API_KEY"] = os.getenv("GEMINI_API_KEY")


## Summarization Middleware

### Summarization MiddleWare
Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:
- Long-running conversations that exceed context windows.
- Multi-turn dialogues with extensive history.
- Applications where preserving full conversation context matters.

In [4]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage,SystemMessage
from langchain_google_genai import ChatGoogleGenerativeAI

## Message based Summarization Middleware




model = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash"
)

agent = create_agent(
    model=model,
    tools=[],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("messages", 10),
            keep=("messages", 4),
        )
    ],
)



In [5]:
## Run with Thread ID

from logging import config


questions=[
    "What is 2+2?",
        "What is 10*5?",
        "What is 100/4?",
        "What is 15-7?",
        "What is 3*3?",
        "What is 4*4?",
]

for q in questions:
    ## Run with Thread ID

    config = {
        "configurable": {
            "thread_id": "test-1"
        }
    }

    questions = [
        "What is 2+2?",
        "What is 10*5?",
        "What is 100/4?",
        "What is 15-7?",
        "What is 3*3?",
        "What is 4*4?",
    ]

    for q in questions:
        response = agent.invoke(
            {"messages": [HumanMessage(content=q)]},
            config=config
        )
        print(f"Messages: {response['messages']}")
        print(f"Messages Length: {len(response['messages'])}")

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


GoogleRateLimitError: Error calling model 'gemini-3.8-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.8-flash\nPlease retry in 11h15m57.59708075s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTie

# Token Size

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage,SystemMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city:str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""


agent=create_agent(
    model=model,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("messages", 10),
            keep=("messages", 4),
        )
    ],
)

config={"configurable":{"thread_id":"test-1"}}


# Token Counter(approximate)

def count_tokens(messages):
    total_chars=sum(len(str(m.content)) for m in messages)
    return total_chars // 4  # Approximate token count (assuming 4 characters per token)

In [ ]:
# Run test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )
    
    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris: ~214 tokens, 4 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='157d8d91-6d86-4a9d-b214-7be983603db7'), AIMessage(content=[], additional_kwargs={'function_call': {'name': 'search_hotels', 'arguments': '{"city": "Paris"}'}, '__gemini_function_call_thought_signatures__': {'call_1067851': 'Eo0CCooCAWkUfRNRL6J+xMNU06CcHYk4o8QG+EPNRD4aUBuE7XEH65IF/6ON7slNb5PWuD+LwnLA7jqPygYOC3FktLWSIZ81fLET6no82bag5ZIx5amXyDeXMR7z+u62+81zrULcy1WBeSW97kguyy8EbSI2RUyfosUiizrPp18cEKjppesON+1opJAcW0Ga9hi5hvOxxlEypHOm5K01eZ5HRCgGoJlcUyXbs9ZT0ZtaY1xROxEjLEGJDG+xbdHUNBQyGptSNXI9Cu6PgWf3K6n3xG46mXlpJHOhXm1+DdY3armS37/ojBfjDEqkERKLOhMJc4nYfA4RD4BlCDXaFRnsynAnZeBpc/gsF/3A0XA='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.8-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a11b36-f192-75f0-84ec-4a5409c08485-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Paris'}, 'id': 'call_1067851', 'type':

GoogleRateLimitError: Error calling model 'gemini-3.8-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.8-flash\nPlease retry in 12h46m30.577171918s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTi

# Fraction

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver


@tool
def search_hotels(city: str) -> str:
    """Search hotels in a city."""
    return f"Hotels in {city}: Grand Hotel $350, City Inn $180, Budget Stay $75"


memory = InMemorySaver()

agent = create_agent(
    model=model,
    tools=[search_hotels],
    checkpointer=memory,
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("tokens", 1000),
            keep=("messages", 4),
        )
    ],
)

config = {
    "configurable": {
        "thread_id": "test-1"
    }
}


cities = [
    "Paris",
    "London",
    "Tokyo",
    "New York",
    "Dubai",
    "Singapore"
]

for city in cities:

    response = agent.invoke(
        {
            "messages": [
                HumanMessage(
                    content=f"Search for hotels in {city}"
                )
            ]
        },
        config=config
    )

    messages = response["messages"]

    print("=" * 60)
    print(f"City: {city}")
    print(f"Messages: {len(messages)}")
    print("Latest:", messages[-1].content)

GoogleRateLimitError: Error calling model 'gemini-3.8-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.8-flash\nPlease retry in 12h40m37.330508922s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTi

# Human In the Loop

Pause agent execution for human approval, editing, or rejection of tool calls before they execute. Human-in-the-loop is useful for the following:
- High-stakes operations requiring human approval (e.g. database writes, financial transactions).
- Compliance workflows where human oversight is mandatory.
- Long-running conversations where human feedback guides the agent.

In [6]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


def read_email_tool(email_id: str) -> str:
    """Simulate reading an email by ID."""
    return f"Email {email_id}: This is a simulated email content."

def send_email_tool(email_id: str, content: str) -> str:
    """Simulate sending an email by ID."""
    return f"Email {email_id} sent with content: {content}"

In [7]:
agent = create_agent(
    model=model,
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"]
                },
                "read_email_tool": False,
            }
        )
    ],
)

In [9]:
# STEP 1: Request
# STEP 1: Request
try:
    response = agent.invoke(
        {
            "messages": [
                HumanMessage(
                    content=(
                        "Send an email to john@rishabh.com "
                        "with subject 'Meeting' and body "
                        "'Let's meet tomorrow at 10 AM.'"
                    )
                )
            ]
        },
        config=config,
    )
    print(response)
except Exception as exc:
    if "RESOURCE_EXHAUSTED" in str(exc) or "429" in str(exc):
        response = None
        print(
            "Gemini API quota exhausted. Wait for the reset period, "
            "use another API key/model, or enable billing."
        )
    else:
        raise

Gemini API quota exhausted. Wait for the reset period, use another API key/model, or enable billing.


In [ ]:
response["messages"]  # This will show the messages exchanged, including any human intervention required.

In [17]:
from langgraph.types import Command
# Step 2: Approve
if "__interrupt__" in response:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {response['messages'][-1].content}")

TypeError: argument of type 'NoneType' is not a container or iterable